# Baseline Solution

Notebook generates a baseline for further review by solving the classical transportation problem.  Model formulation and other data processing functionality can be found in the **model_utils.py** module.

In [2]:
import pandas as pd
import pyomo.environ as pe

import model_utils as utils

The following cell generates the linear programming model.  The `False` parameter builds the model without the addition of any constraints to enforce a store to only be supplied by a single distribution center.  The resulting linear programming model is shown below.  

*Sets*  
Let D be the set of distribution centers.  $D = \{d_1, d_2, d_3, ...\}$  
Let S be the set of retail stores.  $S = \{s_1, s_2, s_3, ...\}$  
Let P be the set of products.  $P = \{p_1, p_2, p_3, ...\}$  

*Params*

Amount of inventory of product p at distribution center d

$$ a_{d,p} \in \mathbb{Z} \quad \forall d \in D, \forall p \in P $$

Demand of product p at store s

$$ b_{s,p} \in \mathbb{Z} \quad \forall s \in S, \forall p \in P $$

Unit cost of shipping product p from distribution center d to store s

$$ c_{d,s,p} \in \mathbb{R} \quad \forall d \in D, \forall s \in S, \forall p \in P $$

*Variables*  

Quantity of product p to ship from distribution center d to store s  

$$x_{d,s,p} \in \mathbb{R} \hspace{2mm} \forall d \in D, \forall s \in S, \forall p \in P$$

*Model*

$$
\begin{array}{cccccl} \text{min} & \sum_{d \in D} \sum_{s \in S} \sum_{p \in P} x_{d,s,p}c_{d,s,p} & & & \\\\
 \text{s.t.} & & & & \\\\
 & \sum_{d \in D} x_{d,s,p} & = & b_{s,p} & \forall s \in S, \forall p \in P & \text{Satisfy store demand} \\\\
 & \sum_{s \in S} x_{d,s,p} & \le & a_{d,p} & \forall d \in D, \forall p \in P & \text{Inventory limit at distribution center} \\\\
 & x_{d,s,p} & \ge & 0 & \forall d \in D, \forall s \in S, \forall p \in P
\end{array}
$$

In [3]:
model = utils.gen_model(False)

In [4]:
solver = pe.SolverFactory('cbc')
results = solver.solve(model, tee=True)

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -printingOptions all -import /tmp/tmpw3zecf33.pyomo.lp -stat=1 -solve -solu /tmp/tmpw3zecf33.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
Presolve 936 (-119) rows, 5436 (-714) columns and 10872 (-1428) elements
Statistics for presolved model


Problem has 936 rows, 5436 columns (5436 with objective) and 10872 elements
Column breakdown:
5436 of type 0.0->inf, 0 of type 0.0->up, 0 of type lo->inf, 
0 of type lo->up, 0 of type free, 0 of type fixed, 
0 of type -inf->0.0, 0 of type -inf->up, 0 of type 0.0->1.0 
Row breakdown:
0 of type E 0.0, 6 of type E 1.0, 0 of type E -1.0, 
900 of type E other, 0 of type G 0.0, 0 of type G 1.0, 
0 of type G other, 0 of type L 0.0, 0 of type L 1.0, 
30 of type L other, 0 of type Range 0.0->1.0, 0 of type Range other, 
0 of type Free 
Presolve 936 (-119) rows, 5436 (-714) columns and 10872 (-1428) elements
0  Obj

## Solution Review

The below cell shows the optimal solution yields a transportation cost of $5,697.04.  The following cell creates an annotated view of the resulting plan stored in a Pandas dataframe.

In [5]:
print(f"Transportation Cost: ${pe.value(model.obj):,.2f}")

Transportation Cost: $5,697.04


In [6]:
plan_df = utils.annotate_model_plan(model)
plan_df.head()

,product,product_desc,dist_center,dist_center_loc,store,store_city_loc,qty
0,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",2023,"Alabaster, AL",18.0
1,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",1261,"Albertville, AL",79.0
2,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",5171,"Alexandria, AL",50.0
3,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",1115,"Anniston, AL",62.0
4,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",1208,"Anniston, AL",40.0


As discussed previously, there exists many times where a store is satisfied by multiple distribution centers.  The below cell identifies these stores.  It is worth noting that over 50% of stores have at least 2 distribution centers being used to meet their demand.

In [24]:
store_dc_counts = plan_df.groupby('store')['dist_center'].nunique()
multi_dc_stores = store_dc_counts[store_dc_counts > 1].index.tolist()
multi_dc_store_df = plan_df[plan_df['store'].isin(multi_dc_stores)].sort_values(by=['store','dist_center'])
print(f"Qty. stores with multiple distribution center deliveries: {len(multi_dc_stores)} ({(len(multi_dc_stores)/plan_df['store'].nunique())*100:.1f}%)" )
multi_dc_store_df[:25]

Qty. stores with multiple distribution center deliveries: 118 (57.6%)


,product,product_desc,dist_center,dist_center_loc,store,store_city_loc,qty
169,AS242,Prestone Windshield De-Icer,18,"Greensboro, NC",1016,"Knoxville, TN",59.0
355,AS240,Prestone 15.5 Ounce Windshield De-Icer,18,"Greensboro, NC",1016,"Knoxville, TN",62.0
541,16024,Hopkins SubZero 21-1/2 Inch Snow Brush,18,"Greensboro, NC",1016,"Knoxville, TN",54.0
865,20,Scotwood Industries Traction Melt Driveway De-...,18,"Greensboro, NC",1016,"Knoxville, TN",32.0
703,17228,Hopkins SubZero Auto Emergency Shovel,22,"Lebanon, TN",1016,"Knoxville, TN",39.0
63,AS242,Prestone Windshield De-Icer,2,"Saraland, AL",1021,"Rome, GA",116.0
278,AS240,Prestone 15.5 Ounce Windshield De-Icer,7,"Forest Park, GA",1021,"Rome, GA",83.0
465,16024,Hopkins SubZero 21-1/2 Inch Snow Brush,7,"Forest Park, GA",1021,"Rome, GA",104.0
649,17228,Hopkins SubZero Auto Emergency Shovel,7,"Forest Park, GA",1021,"Rome, GA",22.0
794,20,Scotwood Industries Traction Melt Driveway De-...,7,"Forest Park, GA",1021,"Rome, GA",70.0
